# RDD Caching and Persistence Strategies

**Module**: 02_rdd  
**Notebook**: 05_rdd_caching_and_persistence.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How RDD caching and persistence work, comparing storage levels (MEMORY_ONLY, MEMORY_AND_DISK, MEMORY_ONLY_SER), and cache eviction.
- **Why does this topic exist?**: Caching prevents redundant recomputation across multiple actions, turning 10-minute pipelines into 10-second queries.
- **What problem does it solve?**: Blind caching wastes executor RAM, causes memory spill to disk, and chokes shuffle execution memory.
- **Where is it used in real-world Data Engineering?**: Iterative machine learning algorithms (K-Means, Gradient Descent) and branching ETL pipelines.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Important |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | RDD lineage, memory basics |
| **Required depth** | Working knowledge: can choose a storage level and verify cache behavior |
| **Data Engineer** | Use `cache()`/`persist()` correctly, inspect the Storage tab, and avoid caching one-pass data. |
| **Senior Data Engineer** | Reason about memory pressure, eviction, serialization, and when caching hurts. |
| **Spark specialist** | Explain BlockManager behavior and storage-level trade-offs. |
| **Next topic** | `03_dataframe_and_spark_sql/01_dataframe_fundamentals.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Caching is often described as "putting data in memory", which is misleading. A cached partition may be in
memory, on disk, serialized, or evicted and recomputed. Caching a one-pass DataFrame adds memory pressure with no
benefit; caching a reused intermediate result can remove repeated scans and shuffles. The decision should be based
on reuse and measured cost.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **`cache()`** | Mark a dataset for reuse | RDD: MEMORY_ONLY; DataFrame: MEMORY_AND_DISK by default |
| **`persist(level)`** | Choose a storage level | Memory, disk, serialized, off-heap, replication |
| **StorageLevel** | Where and how blocks are stored | MEMORY_ONLY, MEMORY_AND_DISK, DISK_ONLY, *_SER, OFF_HEAP |
| **Eviction** | Remove blocks under memory pressure | LRU-like behavior in the memory store |
| **BlockManager** | The block storage service | Driver and executor component for cached/shuffle/broadcast blocks |
| **`unpersist()`** | Release cached blocks | Frees memory and disk |

### How it works, step by step

1. `cache()`/`persist()` is lazy; the first action materializes blocks.
2. The BlockManager stores partitions on the executor that computed them.
3. The driver's BlockManagerMaster tracks block locations.
4. If memory is insufficient, blocks may spill to disk or be evicted.
5. A later action reads cached blocks when available; otherwise it recomputes.
6. `unpersist()` removes the blocks and the cache metadata.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Storage Levels Comparison

| Storage Level | Space Used | CPU Time | In Memory | On Disk | Serialized | Off-Heap |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **MEMORY_ONLY** | High | Low | Yes | No | No | No |
| **MEMORY_ONLY_SER** | Low | High | Yes | No | Yes | No |
| **MEMORY_AND_DISK** | High | Medium | Some | Some | No | No |
| **MEMORY_AND_DISK_SER** | Low | High | Some | Some | Yes | No |
| **DISK_ONLY** | Low | High | No | Yes | Yes | No |

- `rdd.cache()` is an alias for `rdd.persist(StorageLevel.MEMORY_ONLY)`.
- Always call `rdd.unpersist()` when finished with the cached dataset.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("RDD_Caching_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Hands-On: Caching an Expensive RDD Computation

In [ ]:
from pyspark import StorageLevel

sc = spark.sparkContext
orders_rdd = sc.textFile("/opt/spark-data/retail/orders.csv")

# Perform an expensive parsing step
parsed_orders = (
    orders_rdd
    .filter(lambda l: not l.startswith("order_id"))
    .map(lambda l: l.split(","))
    .map(lambda c: (c[7], float(c[5]) * int(c[4]))) # (status, total_price)
)

# Persist in memory
parsed_orders.persist(StorageLevel.MEMORY_ONLY)
print(f"Is RDD cached?: {parsed_orders.is_cached}")
print(f"Storage Level:  {parsed_orders.getStorageLevel()}")

## 4. Benchmarking Cached vs Uncached Performance

Watch the execution time drop dramatically on the second Action!

In [ ]:
# Action 1: First time evaluates lineage AND caches into BlockManager
t0 = time.time()
count_1 = parsed_orders.count()
t_action1 = time.time() - t0

# Action 2: Evaluates directly from in-memory BlockManager!
t0 = time.time()
count_2 = parsed_orders.count()
t_action2 = time.time() - t0

print(f"Action 1 (Lineage + Cache populate): {t_action1:.4f}s")
print(f"Action 2 (Directly from RAM cache):  {t_action2:.4f}s")
print(f"Speedup from caching:                 {t_action1 / t_action2:.2f}x")

## 5. Unpersisting to Free Memory

Always unpersist when your branching pipeline finishes!

In [ ]:
parsed_orders.unpersist()
print(f"Is RDD cached after unpersist?: {parsed_orders.is_cached}")

---

## Experiment: Inspecting Cache in Spark UI Storage Tab

- **1. What are we testing?**: Persist an RDD, trigger an action, and verify that cached partitions appear in the Spark UI Storage tab.
- **2. What do we expect?**: Storage tab shows memory used and percentage cached.

In [ ]:
rdd_sample = sc.parallelize(range(100000), 4).map(lambda x: x * 3)
rdd_sample.persist(StorageLevel.MEMORY_ONLY)
# Action to populate cache
rdd_sample.count()

print("Check http://localhost:4040/storage/ to see the cached RDD blocks!")
print(f"Storage level: {rdd_sample.getStorageLevel()}")
rdd_sample.unpersist()

**3. What happened?**: Cached blocks were populated in the BlockManager and visible in Spark UI.  
**4. Why did it happen?**: Calling an action materializes the RDD partitions and stores them in the Storage Memory pool.  
**5. Practical Engineering Lesson**: Always monitor the Storage tab in Spark UI to see how much cached data is in memory and whether it spills to disk.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Iterative machine-learning feature preparation

- **Industry**: Recommendation systems
- **Scenario**: A recommender repeatedly uses the same user-item feature matrix across many iterations.
- **Problem**: Recomputing the feature join and aggregation on every iteration dominates runtime.
- **Why this concept is relevant**: Caching the reused feature matrix avoids repeated scans and shuffles, if it fits in memory/disk.
- **Implementation idea**: Build the feature DataFrame, `persist(StorageLevel.MEMORY_AND_DISK)`, materialize with `count()`, and unpersist after training.
- **Expected result**: Faster iterations and more predictable runtime.
- **Alternative**: A feature store or a distributed cache outside Spark.
- **Trade-offs**: Spark caching is simple for a single job; a feature store is better for cross-job reuse.

### Use Case 2: Interactive analyst session

- **Industry**: Retail analytics
- **Scenario**: An analyst runs several queries against the same filtered orders dataset.
- **Problem**: Each query re-reads and re-filters the source, slowing exploration.
- **Why this concept is relevant**: Caching the filtered dataset once can make subsequent queries faster.
- **Implementation idea**: Filter, cache, run 3–5 queries, inspect the Storage tab, then unpersist.
- **Expected result**: Faster interactive queries and visible cache size.
- **Alternative**: A warehouse with result caching or a BI extract.
- **Trade-offs**: Spark caching is local to the session; a warehouse cache is shared and managed.

---

## Business Value

- Can reduce repeated I/O and shuffle work.
- Can improve SLA for iterative or interactive workloads.
- Can increase memory pressure and GC if overused.
- Requires monitoring and cleanup to avoid wasted capacity.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Cache only datasets reused by multiple actions.
- Materialize the cache with an action and verify it in the Storage tab.
- Use `MEMORY_AND_DISK` when the dataset may not fit; expect disk I/O.
- Unpersist when the reuse window ends.
- Do not cache a DataFrame that is immediately written once.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Cache hit vs recomputation

**Steps**

1. Create a filtered DataFrame from `orders.csv`.
2. Time the first `count()` and a second `count()` without caching.
3. Cache, materialize, and time two more `count()` calls.
4. Open the Storage tab and compare memory/disk size.

**What to observe**: The cached second action should avoid the source scan, but the first action pays the materialization cost.

**Healthy result**: You can explain the break-even point and see the cached blocks.

**Problematic result**: Caching a one-pass dataset adds cost without a later reuse.

**Engineering takeaway**: Cache based on reuse, not habit.

---

## Trade-offs and Alternatives

### Advantages

- Avoids recomputation
- Simple API
- Works for RDDs and DataFrames

### Limitations

- Consumes memory/disk
- Can evict other useful blocks
- Does not guarantee all data stays in RAM

### When to use it

- Reused intermediate results
- Iterative algorithms
- Interactive analysis

### When not to use it

- One-pass pipelines
- Datasets much larger than memory without a plan
- Forgetting to unpersist

### Alternatives inside Spark

- `persist`/`unpersist`
- Storage tab
- AQE

### Alternatives outside Spark

- Feature stores
- Warehouse result caches
- Redis/Memcached for serving

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Persist with MEMORY_AND_DISK
**Problem Statement**: Persist parsed_orders using StorageLevel.MEMORY_AND_DISK, run count(), and verify getStorageLevel().  
**Input / Context**: StorageLevel.MEMORY_AND_DISK

In [ ]:
# Exercise 1: MEMORY_AND_DISK persistence

### Exercise 2 (Intermediate): Compare Cached vs Uncached Branching Aggregations
**Problem Statement**: Execute sum() and max() on an unpersisted RDD vs cached RDD and compare total execution time.  
**Input / Context**: sc.parallelize(range(1000000), 4).map(lambda x: x**2)

In [ ]:
# Exercise 2: Branching actions benchmark

### Exercise 3 (Advanced): Inspect Serialized Storage Level Memory Footprint
**Problem Statement**: Persist an RDD with MEMORY_ONLY and MEMORY_ONLY_SER and compare behavior.  
**Input / Context**: StorageLevel.MEMORY_ONLY_SER

In [ ]:
# Exercise 3: Serialized storage

---

## Production Debugging Exercise

### Cache that never helps

**Scenario**: A job caches a DataFrame, writes it once, and never reads it again; the job uses more memory and runs slower.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Count how many actions consume the DataFrame.
2. Check the Storage tab for cached blocks that are never read.
3. Remove the cache or move it after a reused intermediate step.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Persist with MEMORY_AND_DISK):
```python
p = sc.parallelize(range(100)).persist(StorageLevel.MEMORY_AND_DISK)
p.count()
print("Storage level:", p.getStorageLevel())
p.unpersist()
```

#### Solution 2 (Compare Cached vs Uncached Branching Aggregations):
```python
r = sc.parallelize(range(500000), 4).map(lambda x: x * 2).persist()
# Warm up cache with action
r.count()
t0 = time.time()
s_val = r.sum()
m_val = r.max()
print(f"Sum: {s_val}, Max: {m_val} executed in {time.time()-t0:.4f}s from cache")
r.unpersist()
```

#### Solution 3 (Inspect Serialized Storage Level Memory Footprint):
```python
r_ser = sc.parallelize([f"Str_{i}"for i in range(50000)]).persist(StorageLevel.MEMORY_ONLY_SER)
r_ser.count()
print("Serialized level:", r_ser.getStorageLevel())
r_ser.unpersist()
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **When to Cache**: Cache only when an expensive RDD is reused across 2 or more actions.
- **unpersist() Discipline**: Always release memory when finished to avoid choking execution pools.
- **Storage Levels**: MEMORY_ONLY is fastest; MEMORY_AND_DISK prevents recomputation if RAM is full.

### Essential Technical Terms
- `cache()`: Persist with default MEMORY_ONLY (RDD) or MEMORY_AND_DISK (DataFrame).
- `StorageLevel`: Configuration defining memory, disk, serialization, and replication flags.
- `unpersist()`: Eagerly removes cached blocks from the BlockManager.

### Common Engineering Mistakes
- **Caching Single-Use RDDs**: Adds serialization and BlockManager overhead with zero reuse benefit.
- **Forgetting to Unpersist**: Leads to memory eviction and disk spilling in subsequent jobs.

### Interview Questions on This Topic
- **Q: What is the difference between cache() and persist()?**
  - *A*: cache() uses the default storage level; persist() accepts a custom StorageLevel parameter.
- **Q: Why might caching make a job slower?**
  - *A*: If the data is only used once, caching incurs storage overhead without any recomputation savings.




### More Interview Questions

- **Q: What is the default storage level for DataFrame `cache()`?**
  - *A*: In this stack, DataFrame `cache()` uses the dataset default, typically `MEMORY_AND_DISK`; RDD `cache()` uses `MEMORY_ONLY`.
- **Q: Does caching guarantee that all data stays in RAM?**
  - *A*: No. Partitions may be serialized, spilled to disk, evicted, or recomputed depending on the storage level and memory pressure.

### What to Study Next
Module 02 complete! Proceed to Module 03: DataFrames and Spark SQL.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')